In [1]:
import os

In [2]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification\\research'

In [3]:
os.chdir("../")

%pwd

In [4]:
%pwd

'd:\\projects\\project-9 kidney disease\\kidney-desease-classification_again\\kidney-classification'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzipped_data_dir: Path
    

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [12]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
        
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzipped_data_dir=config.unzipped_data_dir
        )
        
        return data_ingestion_config

In [8]:
import zipfile
import gdown
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
        
    def download_file(self)-> str:
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs(self.config.root_dir, exist_ok = True)
            logger.info(f"downloading data from {dataset_url} into file {zip_download_dir}")
            
            file_id = dataset_url.split('/')[-2]
            gdown.download(id=file_id, output=zip_download_dir)
            size = get_size(Path(zip_download_dir))
            logger.info(f"file downloaded successfully and saved at {zip_download_dir} with size {size}")
            return zip_download_dir
        except Exception as e:
            raise e
    
    
    def extract_zip_file(self, zip_file_path: str):
        try:
            logger.info(f"extraction of file: {zip_file_path} started")
            unzip_dir = self.config.unzipped_data_dir
            os.makedirs(unzip_dir, exist_ok = True)
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_dir)
            logger.info(f"extraction completed successfully at location: {unzip_dir}")
        except Exception as e:
            raise e

In [13]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    
    data_ingestion = DataIngestion(config = data_ingestion_config)
    zip_file_path = data_ingestion.download_file()
    data_ingestion.extract_zip_file(zip_file_path=zip_file_path)
except Exception as e:
    raise e

[2026-04-06 06:28:29,085: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-06 06:28:29,096: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-06 06:28:29,100: INFO: common: created directory at : artifacts]
[2026-04-06 06:28:29,110: INFO: common: created directory at : artifacts/data_ingestion]
[2026-04-06 06:28:29,115: INFO: 2222662554: downloading data from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=a860bfff-6b2e-486a-8c6d-1e81964bc982
To: d:\projects\project-9 kidney disease\kidney-desease-classification_again\kidney-classification\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:13<00:00, 4.13MB/s]

[2026-04-06 06:28:50,559: INFO: 2222662554: file downloaded successfully and saved at artifacts/data_ingestion/data.zip with size ~ 56361 KB]
[2026-04-06 06:28:50,565: INFO: 2222662554: extraction of file: artifacts/data_ingestion/data.zip started]


[2026-04-06 06:28:54,039: INFO: 2222662554: extraction completed successfully at location: artifacts/data_ingestion]
